# BERTimbau NER for Cellphone Product Titles

This notebook fine-tunes **BERTimbau** (`neuralmind/bert-base-portuguese-cased`), a
BERT model pretrained on Brazilian Portuguese, for token classification (NER) on
e-commerce product titles (cellphones/accessories). It is one of three NER
techniques being compared for this assignment (the other two: spaCy and CRF :
are covered in `01_spacy_ner.ipynb` and `02_crf_ner.ipynb`).

Pipeline:
1. Load the char-offset span annotations (`train.jsonl` / `test.jsonl`).
2. Tokenize with BERTimbau's WordPiece tokenizer and align char spans to
   subword-level BIO labels (standard HF NER convention: label only the first
   subword of each token, `-100` for continuation subwords and special tokens).
3. Fine-tune `AutoModelForTokenClassification` with `transformers.Trainer`.
4. Evaluate on the held-out test set with `seqeval` (overall + per-tag
   precision/recall/F1).
5. Inspect qualitative predictions.

This machine is CPU-only, so the whole run (240 train examples, 8 epochs,
batch size 8) is tuned to finish in a few minutes.


## Setup

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if not (ROOT / "scripts").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "scripts"))

import json
import time

import numpy as np
import torch
from datasets import Dataset
from seqeval.metrics import classification_report
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
)

import train_bertimbau as bt

print("Model:", bt.MODEL_NAME)
print("Tags:", bt.TAGS)


/Users/joaorietra/Developer/mt-lab-session-ner-starter/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Model: neuralmind/bert-base-portuguese-cased
Tags: ['TIPO', 'MARCA', 'MODELO', 'MEMORIA', 'RAM', 'COR', 'TELA']


## 1. Load data

Each line of `train.jsonl` / `test.jsonl` is `{"text": ..., "entities": [[start, end, "TAG"], ...]}`
: character-offset span annotations.


In [2]:
train_examples = bt.load_jsonl(bt.TRAIN_PATH)
test_examples = bt.load_jsonl(bt.TEST_PATH)
print(f"Train: {len(train_examples)} examples | Test: {len(test_examples)} examples")
train_examples[0]


Train: 240 examples | Test: 60 examples


{'text': 'Smartphone Samsung Galaxy A10s 32GB Android 9.0 Tela 6.2” Octa-Core 4G Câmera 13MP+2MP - Preto',
 'entities': [[0, 10, 'TIPO'],
  [11, 18, 'MARCA'],
  [19, 30, 'MODELO'],
  [31, 35, 'MEMORIA'],
  [48, 57, 'TELA'],
  [89, 94, 'COR']]}

## 2. Tokenization + label alignment

`bt.align_labels` (imported from `scripts/train_bertimbau.py`, the single source
of truth also used by the standalone script) does the core work:

1. Tokenizes each title with BERTimbau's tokenizer, requesting
   `return_offsets_mapping=True` so every subword token carries its
   `(char_start, char_end)` span in the original text.
2. Uses `tokenized.word_ids(batch_index=i)` to know which subwords belong to
   the same original word (critically **not** `sequence_ids`, which only
   tells you which *sentence* a token belongs to in a pair: a distinction
   worth calling out since it's an easy, silent bug in NER label alignment).
3. For each word's *first* subword, looks up which annotated entity span (if
   any) contains it, and assigns `B-<TAG>` for the first subword of an entity
   and `I-<TAG>` for subsequent words inside the same entity.
4. Every continuation subword and special token (`[CLS]`, `[SEP]`) gets label
   `-100`, so `CrossEntropyLoss` ignores them during training: the standard
   HF token-classification convention.

Let's see it on one example.


In [3]:
tokenizer = AutoTokenizer.from_pretrained(bt.MODEL_NAME)

demo = train_examples[0]
demo_ds = bt.align_labels([demo], tokenizer)
tokens = tokenizer.convert_ids_to_tokens(demo_ds[0]["input_ids"])
labels = demo_ds[0]["labels"]

print(demo["text"])
print(demo["entities"])
print()
for tok, lab in zip(tokens, labels):
    tag = bt.ID2LABEL[lab] if lab != -100 else "-100 (ignored)"
    print(f"  {tok!r:15} {tag}")


Smartphone Samsung Galaxy A10s 32GB Android 9.0 Tela 6.2” Octa-Core 4G Câmera 13MP+2MP - Preto
[[0, 10, 'TIPO'], [11, 18, 'MARCA'], [19, 30, 'MODELO'], [31, 35, 'MEMORIA'], [48, 57, 'TELA'], [89, 94, 'COR']]

  '[CLS]'         -100 (ignored)
  'S'             B-TIPO
  '##mart'        -100 (ignored)
  '##pho'         -100 (ignored)
  '##ne'          -100 (ignored)
  'Sam'           B-MARCA
  '##su'          -100 (ignored)
  '##ng'          -100 (ignored)
  'Gala'          B-MODELO
  '##xy'          -100 (ignored)
  'A'             I-MODELO
  '##10'          -100 (ignored)
  '##s'           -100 (ignored)
  '32'            B-MEMORIA
  '##GB'          -100 (ignored)
  'Android'       O
  '9'             O
  '.'             O
  '0'             O
  'Tel'           B-TELA
  '##a'           -100 (ignored)
  '6'             I-TELA
  '.'             I-TELA
  '2'             I-TELA
  '”'             I-TELA
  'Oc'            O
  '##ta'          -100 (ignored)
  '-'             O
  'Core'         

In [4]:
train_dataset = bt.align_labels(train_examples, tokenizer)
test_dataset = bt.align_labels(test_examples, tokenizer)
train_dataset, test_dataset


(Dataset({
     features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
     num_rows: 240
 }),
 Dataset({
     features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
     num_rows: 60
 }))

## 3. Model + training setup

We fine-tune `AutoModelForTokenClassification` from `neuralmind/bert-base-portuguese-cased`
(BERTimbau base, ~110M params, ~400MB download). The classification head
(`num_labels = 1 "O" + 7 tags * 2 (B/I) = 15`) is randomly initialized; the
rest of the weights come from the pretrained BERTimbau checkpoint.

Given only 240 tiny training examples and CPU-only hardware, training config
is deliberately modest: 8 epochs, batch size 8, learning rate 3e-5: enough
epochs to converge on such a small dataset without taking more than a few
minutes.


In [5]:
model = AutoModelForTokenClassification.from_pretrained(
    bt.MODEL_NAME,
    num_labels=len(bt.LABEL_LIST),
    id2label=bt.ID2LABEL,
    label2id=bt.LABEL2ID,
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir=str(ROOT / "models" / "bertimbau_run_nb"),  # gitignored, outside repo tracking
    num_train_epochs=8,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=3e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",
    logging_strategy="epoch",
    report_to=[],
    use_cpu=True,
    seed=42,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=data_collator,
    compute_metrics=bt.compute_metrics_builder(),
    processing_class=tokenizer,
)


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 24240.26it/s]


[transformers] BertForTokenClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/archi

## 4. Fine-tuning

This actually trains the model end-to-end: expect a few minutes on CPU.

In [6]:
start = time.time()
trainer.train()
elapsed = time.time() - start
print(f"Training finished in {elapsed:.1f}s")


Epoch,Training Loss,Validation Loss,F1
1,1.516997,0.689205,0.801471
2,0.529688,0.330430,0.840871
3,0.275956,0.281904,0.841244
4,0.179558,0.241338,0.865772
5,0.124096,0.247285,0.892675
6,0.097467,0.228489,0.921233
7,0.080626,0.234911,0.914089
8,0.062295,0.241129,0.915952


Training finished in 157.3s


## 5. Evaluation on the test set

We run predictions on the held-out `test.jsonl` (60 examples), drop the
`-100` (continuation/special) positions exactly as during training, and
compute precision/recall/F1 with `seqeval.metrics.classification_report` :
overall (micro/macro/weighted) and per tag.


In [7]:
predictions_output = trainer.predict(test_dataset)
predictions = np.argmax(predictions_output.predictions, axis=2)
labels = predictions_output.label_ids

true_predictions = [
    [bt.ID2LABEL[p] for p, l in zip(pred, label) if l != -100]
    for pred, label in zip(predictions, labels)
]
true_labels = [
    [bt.ID2LABEL[l] for p, l in zip(pred, label) if l != -100]
    for pred, label in zip(predictions, labels)
]

report_str = classification_report(true_labels, true_predictions, digits=4)
report_dict = classification_report(true_labels, true_predictions, digits=4, output_dict=True)
print(report_str)


              precision    recall  f1-score   support

         COR     0.9500    0.8636    0.9048        44
       MARCA     0.9219    0.9833    0.9516        60
     MEMORIA     0.9706    1.0000    0.9851        33
      MODELO     0.8382    0.8507    0.8444        67
         RAM     1.0000    1.0000    1.0000        14
        TELA     0.8500    0.8095    0.8293        21
        TIPO     0.9423    0.9423    0.9423        52

   micro avg     0.9144    0.9175    0.9160       291
   macro avg     0.9247    0.9214    0.9225       291
weighted avg     0.9146    0.9175    0.9155       291



In [8]:
bt.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
metrics_payload = {
    "model_name": bt.MODEL_NAME,
    "training_time_seconds": elapsed,
    "num_train_examples": len(train_examples),
    "num_test_examples": len(test_examples),
    "report": report_dict,
}

def to_jsonable(obj):
    if isinstance(obj, dict):
        return {k: to_jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_jsonable(v) for v in obj]
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    return obj

metrics_path = bt.RESULTS_DIR / "bertimbau_metrics.json"
with metrics_path.open("w", encoding="utf-8") as f:
    json.dump(to_jsonable(metrics_payload), f, ensure_ascii=False, indent=2)
print(f"Saved metrics to {metrics_path}")


Saved metrics to /Users/joaorietra/Developer/mt-lab-session-ner-starter/notebooks/results/bertimbau_metrics.json


## 6. Qualitative examples

For a few test titles, reconstruct predicted entity spans from the model's
token-level predictions and compare them against the gold annotations. Since
only the first subword of each word carries a real (trained) label, we
extend that label's span to cover the whole word (first subword's start to
last subword's end) before rebuilding entities: otherwise multi-subword
words like "Smartphone" would be truncated to just "S".


In [9]:
def spans_from_bio(text, tokens_offsets, bio_labels):
    spans = []
    current = None
    for (start, end), label in zip(tokens_offsets, bio_labels):
        if label == "O":
            if current:
                spans.append(current)
                current = None
            continue
        prefix, tag = label.split("-", 1)
        if prefix == "B" or current is None or current[2] != tag:
            if current:
                spans.append(current)
            current = [start, end, tag]
        else:
            current[1] = end
    if current:
        spans.append(current)
    return [(s, e, t, text[s:e]) for s, e, t in spans]


model.eval()
n_qual = min(8, len(test_examples))
qual_examples = []

for i in range(n_qual):
    ex = test_examples[i]
    text = ex["text"]
    enc = tokenizer(text, truncation=True, max_length=64, return_offsets_mapping=True)
    offsets = enc.pop("offset_mapping")
    word_ids = enc.word_ids(0)

    with torch.no_grad():
        logits = model(**{k: torch.tensor([v]) for k, v in enc.items()}).logits
    pred_ids = logits.argmax(-1)[0].tolist()

    word_spans, word_label = {}, {}
    for idx, w in enumerate(word_ids):
        if w is None:
            continue
        start, end = offsets[idx]
        if w not in word_spans:
            word_spans[w] = [start, end]
            word_label[w] = bt.ID2LABEL[pred_ids[idx]]
        else:
            word_spans[w][1] = end

    filtered_offsets = [tuple(word_spans[w]) for w in sorted(word_spans)]
    filtered_labels = [word_label[w] for w in sorted(word_spans)]

    pred_spans = spans_from_bio(text, filtered_offsets, filtered_labels)
    gold_spans = [(s, e, t, text[s:e]) for s, e, t in ex["entities"]]

    print(f"Text: {text}")
    print(f"  Gold: {gold_spans}")
    print(f"  Pred: {pred_spans}")
    print()

    qual_examples.append({"text": text, "gold": gold_spans, "pred": pred_spans})

qual_path = bt.RESULTS_DIR / "bertimbau_qualitative.json"
with qual_path.open("w", encoding="utf-8") as f:
    json.dump(qual_examples, f, ensure_ascii=False, indent=2)
print(f"Saved qualitative examples to {qual_path}")


Text: Smartphone Nokia C20 32GB 4G com 90 dias de internet grátis* - NK081
  Gold: [(0, 10, 'TIPO', 'Smartphone'), (11, 16, 'MARCA', 'Nokia'), (17, 20, 'MODELO', 'C20'), (21, 25, 'MEMORIA', '32GB')]
  Pred: [(0, 10, 'TIPO', 'Smartphone'), (11, 16, 'MARCA', 'Nokia'), (17, 20, 'MODELO', 'C20'), (21, 25, 'MEMORIA', '32GB')]

Text: KIT MOTORISTA 01 - WI392
  Gold: [(0, 3, 'TIPO', 'KIT'), (4, 16, 'MODELO', 'MOTORISTA 01')]
  Pred: [(0, 3, 'TIPO', 'KIT')]

Text: iPhone 12 Pro Apple Dourado 512GB Desbloqueado - MGMW3BZ/A
  Gold: [(0, 13, 'MODELO', 'iPhone 12 Pro'), (14, 19, 'MARCA', 'Apple'), (20, 27, 'COR', 'Dourado'), (28, 33, 'MEMORIA', '512GB')]
  Pred: [(0, 13, 'MODELO', 'iPhone 12 Pro'), (14, 19, 'MARCA', 'Apple'), (20, 27, 'COR', 'Dourado'), (28, 33, 'MEMORIA', '512GB')]

Text: iPhone 12 Pro Apple Azul Pacífico 256GB Desbloqueado - MGMT3BZ/A
  Gold: [(0, 13, 'MODELO', 'iPhone 12 Pro'), (14, 19, 'MARCA', 'Apple'), (20, 33, 'COR', 'Azul Pacífico'), (34, 39, 'MEMORIA', '256GB')]
  Pred: [

Text: Capa Protetora para iPhone 7 Plus Apple, Silicone Vermelha - MMQV2ZM/A
  Gold: [(0, 14, 'TIPO', 'Capa Protetora'), (20, 33, 'MODELO', 'iPhone 7 Plus'), (34, 39, 'MARCA', 'Apple'), (50, 58, 'COR', 'Vermelha')]
  Pred: [(0, 14, 'TIPO', 'Capa Protetora'), (20, 33, 'MODELO', 'iPhone 7 Plus'), (34, 39, 'MARCA', 'Apple'), (50, 58, 'COR', 'Vermelha')]



Text: Smartphone Samsung Galaxy A03s 64GB 4GB RAM Tela 6,5
  Gold: [(0, 10, 'TIPO', 'Smartphone'), (11, 18, 'MARCA', 'Samsung'), (19, 30, 'MODELO', 'Galaxy A03s'), (31, 35, 'MEMORIA', '64GB'), (36, 43, 'RAM', '4GB RAM'), (44, 52, 'TELA', 'Tela 6,5')]
  Pred: [(0, 10, 'TIPO', 'Smartphone'), (11, 18, 'MARCA', 'Samsung'), (19, 30, 'MODELO', 'Galaxy A03s'), (31, 35, 'MEMORIA', '64GB'), (36, 43, 'RAM', '4GB RAM'), (44, 52, 'TELA', 'Tela 6,5')]

Text: Celular Multilaser Vita 3G Dual Chip Tela 1,8 Polegadas + Base Carregadora P9091 - Preto
  Gold: [(0, 7, 'TIPO', 'Celular'), (8, 18, 'MARCA', 'Multilaser'), (19, 26, 'MODELO', 'Vita 3G'), (37, 55, 'TELA', 'Tela 1,8 Polegadas'), (58, 74, 'TIPO', 'Base Carregadora'), (83, 88, 'COR', 'Preto')]
  Pred: [(0, 7, 'TIPO', 'Celular'), (8, 18, 'MARCA', 'Multilaser'), (19, 26, 'MODELO', 'Vita 3G'), (37, 55, 'TELA', 'Tela 1,8 Polegadas'), (83, 88, 'COR', 'Preto')]

Saved qualitative examples to /Users/joaorietra/Developer/mt-lab-session-ner-starter/noteboo

## Summary

BERTimbau, fine-tuned for 8 epochs on 240 examples (CPU only, no checkpoints
committed to the repo), reaches the overall test-set F1 printed above. See
`notebooks/results/bertimbau_metrics.json` for the full per-tag breakdown and
`notebooks/results/bertimbau_qualitative.json` for the qualitative examples.
